# Project 4 — Milestone 1: Quantization Workload Augmentation

This notebook follows the AccelForge workflow from the lab:
1. define workload YAML (`iteration_space_shape`, `bits_per_value`, `einsums`)
2. load with `af.Spec.from_yaml(...)`
3. run `spec.evaluate_mapping()`
4. compare baseline vs quantized workload

## Milestone 1 tasks (from project_4.pdf)

- Study hardware cost of supporting quantization
- Create a compound component to model quantization area/energy
- Pick a workload and augment the spec with einsums that represent quantization

This notebook focuses on the third bullet using a minimal weight-only NVFP4-style model.

In [1]:
from pathlib import Path
import os
import yaml

try:
    import accelforge as af
    AF_AVAILABLE = True
except Exception as e:
    af = None
    AF_AVAILABLE = False
    print("accelforge import failed:", e)
    print("Notebook will still generate workload/mapping YAML files.")
    print("Run evaluation cells in your AccelForge-enabled environment.")


def find_lab_root() -> Path:
    cwd = Path.cwd()

    # Optional explicit override
    env_root = os.environ.get("AF_LAB4_ROOT")
    if env_root:
        return Path(env_root).expanduser()

    # If launched from lab_4 directly
    if (cwd / "project4_m1_quantization_workload.ipynb").exists() or (cwd / "project4_m1").exists():
        return cwd

    # Common nested layout: <repo>/workspace/lab_4
    nested = cwd / "workspace" / "lab_4"
    if nested.exists():
        return nested

    # Known local absolute path fallbacks
    known_paths = [
        Path("/Users/yichongzhang/Desktop/\u5927\u5b66/\u5927\u4e09\u4e0b/hardware design for AI accelerators/final project/Hardware-for-quantization/workspace/lab_4"),
        Path("/Users/bearxiong/Documents/MIT/Deep_Learning/Final-Project/workspace/lab_4"),
    ]
    for known in known_paths:
        if known.exists():
            return known

    return cwd


ROOT = find_lab_root()
OUT_DIR = ROOT / "project4_m1"
OUT_DIR.mkdir(exist_ok=True)

# Standalone architecture file (no part6 dependency)
ARCH_FILE = OUT_DIR / "arch_minimal.yaml"
arch_text = """
arch:
  nodes:
  - !Memory
    name: DRAM
    size: 99999999999
    leak_power: 0
    area: 0
    total_latency: "ceil(max((read_actions + metadata_read_actions) / 2, (write_actions + metadata_write_actions) / 2))"
    tensors: {keep: ~Intermediates, may_keep: All}
    actions:
    - {name: read, energy: 10.0, bits_per_action: 32, latency: 0}
    - {name: write, energy: 10.0, bits_per_action: 32, latency: 0}
    - {name: metadata_read, energy: 2.0, bits_per_action: 16, latency: 0}
    - {name: metadata_write, energy: 2.0, bits_per_action: 16, latency: 0}
  - !Memory
    name: Buffer
    size: 16384
    leak_power: 0
    area: 0
    total_latency: "ceil(max(total_read_actions / 8, total_write_actions / 8))"
    tensors: {keep: ~DRAM, may_keep: All}
    actions:
    - {name: read, energy: 2.0, bits_per_action: 16, latency: 0}
    - {name: write, energy: 2.0, bits_per_action: 16, latency: 0}
    - {name: metadata_read, energy: 2.0, bits_per_action: 16, latency: 0}
    - {name: metadata_write, energy: 2.0, bits_per_action: 16, latency: 0}
  - !Container
    name: PEArray
    spatial:
    - {name: X, fanout: 4}
  - !Compute
    name: MAC
    leak_power: 0
    area: 0
    actions:
    - {name: compute, energy: 1.0, latency: 1}
""".strip()
ARCH_FILE.write_text(arch_text)


def dump_yaml(path: Path, obj: dict):
    with open(path, "w") as f:
        yaml.safe_dump(obj, f, sort_keys=False)


def evaluate(arch_file: Path, workload_file: Path, mapping_file: Path):
    if not AF_AVAILABLE:
        raise RuntimeError("accelforge is unavailable in this kernel.")
    for fp in [arch_file, workload_file, mapping_file]:
        if not Path(fp).exists():
            raise FileNotFoundError(f"Missing required file: {fp}")
    spec = af.Spec.from_yaml(str(arch_file), str(workload_file), str(mapping_file))
    result = spec.evaluate_mapping()
    return result


print("Working directory:", Path.cwd())
print("Lab root:", ROOT)
print("Output directory:", OUT_DIR)
print("Architecture:", ARCH_FILE)

Working directory: /home/workspace/workspace/lab_4
Lab root: /home/workspace/workspace/lab_4
Output directory: /home/workspace/workspace/lab_4/project4_m1
Architecture: /home/workspace/workspace/lab_4/project4_m1/arch_minimal.yaml


## Step 1: Baseline dense GEMM workload

Start from a simple lab-style workload with one einsum.

In [2]:
baseline_workload = {
    "workload": {
        "iteration_space_shape": {
            "m": "0 <= m < 64",
            "n": "0 <= n < 64",
            "k": "0 <= k < 128",
        },
        "bits_per_value": {
            "A": 16,
            "W": 16,
            "Y": 16,
        },
        "einsums": [
            {
                "name": "MatMul",
                "tensor_accesses": [
                    {"name": "A", "projection": ["m", "k"], "density": 1.0},
                    {"name": "W", "projection": ["n", "k"], "density": 1.0},
                    {"name": "Y", "projection": ["m", "n"], "output": True},
                ],
            }
        ],
    }
}

baseline_workload_file = OUT_DIR / "workload_baseline.yaml"
baseline_mapping_file = OUT_DIR / "mapping_baseline.yaml"

baseline_mapping_text = """
mapping:
  nodes:
  - !Storage
    tensors: [A, W, Y]
    component: DRAM
  - !Temporal
    rank_variable: m
    tile_shape: 8
  - !Temporal
    rank_variable: n
    tile_shape: 8
  - !Temporal
    rank_variable: k
    tile_shape: 8
  - !Storage
    tensors: [A, W, Y]
    component: Buffer
  - !Temporal
    rank_variable: n
    tile_shape: 1
  - !Temporal
    rank_variable: k
    tile_shape: 1
  - !Temporal
    rank_variable: m
    tile_shape: 1
  - !Spatial
    rank_variable: m
    tile_shape: 1
    name: X
    component: PEArray
  - !Compute
    einsum: MatMul
    component: MAC
""".strip()

dump_yaml(baseline_workload_file, baseline_workload)
baseline_mapping_file.write_text(baseline_mapping_text)

print("Wrote baseline workload:", baseline_workload_file)
print("Wrote baseline mapping:", baseline_mapping_file)

Wrote baseline workload: /home/workspace/workspace/lab_4/project4_m1/workload_baseline.yaml
Wrote baseline mapping: /home/workspace/workspace/lab_4/project4_m1/mapping_baseline.yaml


In [3]:
if AF_AVAILABLE and ARCH_FILE.exists() and baseline_mapping_file.exists():
    baseline_result = evaluate(ARCH_FILE, baseline_workload_file, baseline_mapping_file)
    print("Baseline mapping evaluation completed.")
    print(baseline_result)
else:
    print("Skipped baseline evaluate_mapping().")
    print("AF_AVAILABLE:", AF_AVAILABLE)
    print("arch exists:", ARCH_FILE.exists(), ARCH_FILE)
    print("mapping exists:", baseline_mapping_file.exists(), baseline_mapping_file)

Baseline mapping evaluation completed.


## Step 2–6: NVFP4-style weight-only augmentation

We now augment the workload spec with explicit quantization tensors and einsums:
- split `k -> kb, ki` with `ki=16`
- add quantized weights `Wq[n,kb,ki]`
- add block scales `Sw[n,kb]`
- add dequantized weights `Wdq[n,kb,ki]`
- add `DequantW` einsum before the main GEMM
- update GEMM to consume `Wdq`

In [4]:
quant_workload = {
    "workload": {
        "iteration_space_shape": {
            "m": "0 <= m < 64",
            "n": "0 <= n < 64",
            "kb": "0 <= kb < 8",
            "ki": "0 <= ki < 16",
        },
        "bits_per_value": {
            "A": 16,
            "Wq": 4,
            "Sw": 16,
            "Wdq": 16,
            "Y": 16,
        },
        "einsums": [
            {
                "name": "DequantW",
                "tensor_accesses": [
                    {"name": "Wq", "projection": ["n", "kb", "ki"], "density": 1.0},
                    {"name": "Sw", "projection": ["n", "kb"], "density": 1.0},
                    {"name": "Wdq", "projection": ["n", "kb", "ki"], "output": True},
                ],
            },
            {
                "name": "MatMulQ",
                "tensor_accesses": [
                    {"name": "A", "projection": ["m", "kb", "ki"], "density": 1.0},
                    {"name": "Wdq", "projection": ["n", "kb", "ki"], "density": 1.0},
                    {"name": "Y", "projection": ["m", "n"], "output": True},
                ],
            },
        ],
    }
}

quant_workload_file = OUT_DIR / "workload_nvfp4_weight_only.yaml"
dump_yaml(quant_workload_file, quant_workload)
print("Wrote quantized workload:", quant_workload_file)
with open(quant_workload_file) as f:
    print(f.read())

Wrote quantized workload: /home/workspace/workspace/lab_4/project4_m1/workload_nvfp4_weight_only.yaml
workload:
  iteration_space_shape:
    m: 0 <= m < 64
    n: 0 <= n < 64
    kb: 0 <= kb < 8
    ki: 0 <= ki < 16
  bits_per_value:
    A: 16
    Wq: 4
    Sw: 16
    Wdq: 16
    Y: 16
  einsums:
  - name: DequantW
    tensor_accesses:
    - name: Wq
      projection:
      - n
      - kb
      - ki
      density: 1.0
    - name: Sw
      projection:
      - n
      - kb
      density: 1.0
    - name: Wdq
      projection:
      - n
      - kb
      - ki
      output: true
  - name: MatMulQ
    tensor_accesses:
    - name: A
      projection:
      - m
      - kb
      - ki
      density: 1.0
    - name: Wdq
      projection:
      - n
      - kb
      - ki
      density: 1.0
    - name: Y
      projection:
      - m
      - n
      output: true



## Step 8: Validate with AccelForge (`from_yaml` + `evaluate_mapping()`)

We generate a first-pass quantized mapping that includes both einsums:
- `DequantW`
- `MatMulQ`

If mapping evaluation fails due scheduler constraints, the parser check still confirms workload structure validity and you can iterate the mapping next.

In [5]:
quant_mapping_file = OUT_DIR / "mapping_nvfp4_weight_only.yaml"
quant_mapping_text = """
mapping:
  nodes:
  - !Storage
    tensors: [A, Wq, Sw, Wdq, Y]
    component: DRAM
  - !Temporal
    rank_variable: m
    tile_shape: 8
  - !Temporal
    rank_variable: n
    tile_shape: 4
  - !Temporal
    rank_variable: kb
    tile_shape: 2
  - !Temporal
    rank_variable: ki
    tile_shape: 8
  - !Storage
    tensors: [A, Wq, Sw, Wdq, Y]
    component: Buffer
  - !Temporal
    rank_variable: n
    tile_shape: 1
  - !Temporal
    rank_variable: kb
    tile_shape: 1
  - !Temporal
    rank_variable: ki
    tile_shape: 1
  - !Temporal
    rank_variable: m
    tile_shape: 1
  - !Spatial
    rank_variable: m
    tile_shape: 1
    name: X
    component: PEArray
  - !Compute
    einsum: DequantW
    component: MAC
  - !Compute
    einsum: MatMulQ
    component: MAC
""".strip()

quant_mapping_file.write_text(quant_mapping_text)
print("Wrote quantized mapping:", quant_mapping_file)

if AF_AVAILABLE and ARCH_FILE.exists() and quant_workload_file.exists() and quant_mapping_file.exists():
    # Parser validation + mapping evaluation
    spec_q = af.Spec.from_yaml(str(ARCH_FILE), str(quant_workload_file), str(quant_mapping_file))
    print("Quantized spec parsed successfully.")

    try:
        quant_result = spec_q.evaluate_mapping()
        print("Quantized mapping evaluation completed.")
        print(quant_result)
    except Exception as e:
        print("Quantized mapping parse succeeded, but evaluate_mapping() needs mapping refinement:")
        print(type(e).__name__, e)
else:
    print("Skipped Spec.from_yaml/evaluate_mapping().")
    print("AF_AVAILABLE:", AF_AVAILABLE)
    print("arch exists:", ARCH_FILE.exists(), ARCH_FILE)
    print("workload exists:", quant_workload_file.exists(), quant_workload_file)
    print("mapping exists:", quant_mapping_file.exists(), quant_mapping_file)

Wrote quantized mapping: /home/workspace/workspace/lab_4/project4_m1/mapping_nvfp4_weight_only.yaml
Quantized spec parsed successfully.
Quantized mapping evaluation completed.


## Full NVFP4 Quantization (W4A4) — Two-Level Scaling

NVIDIA's NVFP4 on Blackwell uses **two-level scaling**:
- **Per-tensor scale** (coarse): One FP32 scalar per entire tensor. `projection: []`.
- **Per-block scale** (fine): One FP8 (E4M3) scale per block of 16 elements along K.

### Quantization (FP16 → FP4):
1. `TensorScaleA`: $S_{ga} = \text{reduce}_{m,k_b,k_i}(|A|)$ — per-tensor scale (scalar)
2. `TensorQuantA`: $A_{scl} = A \times S_{ga}^{-1}$
3. `BlockScaleA`: $S_{ba}[m,k_b] = \text{reduce}_{k_i}(|A_{scl}|)$ — per-block scale
4. `BlockQuantA`: $A_q = A_{scl} \times S_{ba}^{-1}$ — quantize to FP4
5–8. Same for weights

### FP4 × FP4 Compute (FP32 accumulate):
9. `MatMulNVFP4`: $Y_{raw}[m,n,k_b] \mathrel{+}= A_q \times W_q$ — reduce $k_i$

### Rescale (2 inputs per einsum):
10. `RescaleBlockA`: $Y_{tmp} = Y_{raw} \times S_{ba}[m,k_b]$
11. `RescaleBlockW`: $Y_{blk} = Y_{tmp} \times S_{bw}[n,k_b]$
12. `RescaleTensorA`: $Y_{tmp2}[m,n] = \sum_{k_b} Y_{blk} \times S_{ga}$ — reduce $k_b$
13. `RescaleTensorW`: $Y = Y_{tmp2} \times S_{gw}$


In [ ]:
nvfp4_full_workload = {
    "workload": {
        "iteration_space_shape": {
            "m": "0 <= m < 64",
            "n": "0 <= n < 64",
            "kb": "0 <= kb < 8",
            "ki": "0 <= ki < 16",
        },
        "bits_per_value": {
            # Original full-precision inputs
            "A": 16,         # FP16 activations
            "W": 16,         # FP16 weights
            # Per-TENSOR scales — FP32, one scalar per entire tensor
            "Sga": 32,       # Global scale for activations (scalar)
            "Sgw": 32,       # Global scale for weights (scalar)
            # Tensor-scaled intermediates
            "Ascl": 16,      # Activations after global scaling
            "Wscl": 16,      # Weights after global scaling
            # Per-BLOCK scales — FP8 E4M3
            "Sba": 8,        # Block scale for activations [m, kb]
            "Sbw": 8,        # Block scale for weights [n, kb]
            # Quantized — FP4 E2M1
            "Aq": 4,         # Quantized activations
            "Wq": 4,         # Quantized weights
            # FP32 accumulator
            "Yraw": 32,      # FP4*FP4 partial sums
            "Ytmp": 32,      # After activation block rescale
            "Yblk": 32,      # After weight block rescale
            "Ytmp2": 32,     # After activation tensor rescale
            # Final output
            "Y": 16,         # FP16 after all rescaling
        },
        "einsums": [
            # ===== Activation quantization =====
            # 1. Per-tensor scale: reduce ALL dims -> scalar
            {
                "name": "TensorScaleA",
                "tensor_accesses": [
                    {"name": "A", "projection": ["m", "kb", "ki"], "density": 1.0},
                    {"name": "Sga", "projection": [], "output": True},
                ],
            },
            # 2. Apply per-tensor scale
            {
                "name": "TensorQuantA",
                "tensor_accesses": [
                    {"name": "A", "projection": ["m", "kb", "ki"], "density": 1.0},
                    {"name": "Sga", "projection": [], "density": 1.0},
                    {"name": "Ascl", "projection": ["m", "kb", "ki"], "output": True},
                ],
            },
            # 3. Per-block scale: reduce ki
            {
                "name": "BlockScaleA",
                "tensor_accesses": [
                    {"name": "Ascl", "projection": ["m", "kb", "ki"], "density": 1.0},
                    {"name": "Sba", "projection": ["m", "kb"], "output": True},
                ],
            },
            # 4. Quantize to FP4
            {
                "name": "BlockQuantA",
                "tensor_accesses": [
                    {"name": "Ascl", "projection": ["m", "kb", "ki"], "density": 1.0},
                    {"name": "Sba", "projection": ["m", "kb"], "density": 1.0},
                    {"name": "Aq", "projection": ["m", "kb", "ki"], "output": True},
                ],
            },
            # ===== Weight quantization =====
            # 5. Per-tensor scale: reduce ALL dims -> scalar
            {
                "name": "TensorScaleW",
                "tensor_accesses": [
                    {"name": "W", "projection": ["n", "kb", "ki"], "density": 1.0},
                    {"name": "Sgw", "projection": [], "output": True},
                ],
            },
            # 6. Apply per-tensor scale
            {
                "name": "TensorQuantW",
                "tensor_accesses": [
                    {"name": "W", "projection": ["n", "kb", "ki"], "density": 1.0},
                    {"name": "Sgw", "projection": [], "density": 1.0},
                    {"name": "Wscl", "projection": ["n", "kb", "ki"], "output": True},
                ],
            },
            # 7. Per-block scale: reduce ki
            {
                "name": "BlockScaleW",
                "tensor_accesses": [
                    {"name": "Wscl", "projection": ["n", "kb", "ki"], "density": 1.0},
                    {"name": "Sbw", "projection": ["n", "kb"], "output": True},
                ],
            },
            # 8. Quantize to FP4
            {
                "name": "BlockQuantW",
                "tensor_accesses": [
                    {"name": "Wscl", "projection": ["n", "kb", "ki"], "density": 1.0},
                    {"name": "Sbw", "projection": ["n", "kb"], "density": 1.0},
                    {"name": "Wq", "projection": ["n", "kb", "ki"], "output": True},
                ],
            },
            # ===== FP4 x FP4 MatMul, FP32 accumulate =====
            # 9. Yraw[m,n,kb] += Aq[m,kb,ki] * Wq[n,kb,ki]  (reduce ki)
            {
                "name": "MatMulNVFP4",
                "tensor_accesses": [
                    {"name": "Aq", "projection": ["m", "kb", "ki"], "density": 1.0},
                    {"name": "Wq", "projection": ["n", "kb", "ki"], "density": 1.0},
                    {"name": "Yraw", "projection": ["m", "n", "kb"], "output": True},
                ],
            },
            # ===== Rescale output (2 inputs each) =====
            # 10. Undo activation block scale
            {
                "name": "RescaleBlockA",
                "tensor_accesses": [
                    {"name": "Yraw", "projection": ["m", "n", "kb"], "density": 1.0},
                    {"name": "Sba", "projection": ["m", "kb"], "density": 1.0},
                    {"name": "Ytmp", "projection": ["m", "n", "kb"], "output": True},
                ],
            },
            # 11. Undo weight block scale
            {
                "name": "RescaleBlockW",
                "tensor_accesses": [
                    {"name": "Ytmp", "projection": ["m", "n", "kb"], "density": 1.0},
                    {"name": "Sbw", "projection": ["n", "kb"], "density": 1.0},
                    {"name": "Yblk", "projection": ["m", "n", "kb"], "output": True},
                ],
            },
            # 12. Undo activation tensor scale + reduce kb
            {
                "name": "RescaleTensorA",
                "tensor_accesses": [
                    {"name": "Yblk", "projection": ["m", "n", "kb"], "density": 1.0},
                    {"name": "Sga", "projection": [], "density": 1.0},
                    {"name": "Ytmp2", "projection": ["m", "n"], "output": True},
                ],
            },
            # 13. Undo weight tensor scale
            {
                "name": "RescaleTensorW",
                "tensor_accesses": [
                    {"name": "Ytmp2", "projection": ["m", "n"], "density": 1.0},
                    {"name": "Sgw", "projection": [], "density": 1.0},
                    {"name": "Y", "projection": ["m", "n"], "output": True},
                ],
            },
        ],
    }
}

nvfp4_full_workload_file = OUT_DIR / "workload_nvfp4_full.yaml"
dump_yaml(nvfp4_full_workload_file, nvfp4_full_workload)
print("Wrote full NVFP4 workload:", nvfp4_full_workload_file)
with open(nvfp4_full_workload_file) as f:
    print(f.read())


In [ ]:
nvfp4_full_mapping_file = OUT_DIR / "mapping_nvfp4_full.yaml"
nvfp4_full_mapping_text = """
mapping:
  nodes:
  - !Storage
    tensors: [A, W, Sga, Sgw, Ascl, Wscl, Sba, Sbw, Aq, Wq, Yraw, Ytmp, Yblk, Ytmp2, Y]
    component: DRAM
  - !Temporal
    rank_variable: m
    tile_shape: 8
  - !Temporal
    rank_variable: n
    tile_shape: 4
  - !Temporal
    rank_variable: kb
    tile_shape: 2
  - !Temporal
    rank_variable: ki
    tile_shape: 8
  - !Storage
    tensors: [A, W, Sga, Sgw, Ascl, Wscl, Sba, Sbw, Aq, Wq, Yraw, Ytmp, Yblk, Ytmp2, Y]
    component: Buffer
  - !Temporal
    rank_variable: n
    tile_shape: 1
  - !Temporal
    rank_variable: kb
    tile_shape: 1
  - !Temporal
    rank_variable: ki
    tile_shape: 1
  - !Temporal
    rank_variable: m
    tile_shape: 1
  - !Spatial
    rank_variable: m
    tile_shape: 1
    name: X
    component: PEArray
  - !Compute
    einsum: TensorScaleA
    component: MAC
  - !Compute
    einsum: TensorQuantA
    component: MAC
  - !Compute
    einsum: BlockScaleA
    component: MAC
  - !Compute
    einsum: BlockQuantA
    component: MAC
  - !Compute
    einsum: TensorScaleW
    component: MAC
  - !Compute
    einsum: TensorQuantW
    component: MAC
  - !Compute
    einsum: BlockScaleW
    component: MAC
  - !Compute
    einsum: BlockQuantW
    component: MAC
  - !Compute
    einsum: MatMulNVFP4
    component: MAC
  - !Compute
    einsum: RescaleBlockA
    component: MAC
  - !Compute
    einsum: RescaleBlockW
    component: MAC
  - !Compute
    einsum: RescaleTensorA
    component: MAC
  - !Compute
    einsum: RescaleTensorW
    component: MAC
""".strip()

nvfp4_full_mapping_file.write_text(nvfp4_full_mapping_text)
print("Wrote full NVFP4 mapping:", nvfp4_full_mapping_file)

if AF_AVAILABLE and ARCH_FILE.exists() and nvfp4_full_workload_file.exists() and nvfp4_full_mapping_file.exists():
    spec_nvfp4 = af.Spec.from_yaml(str(ARCH_FILE), str(nvfp4_full_workload_file), str(nvfp4_full_mapping_file))
    print("Full NVFP4 spec parsed successfully.")

    try:
        nvfp4_full_result = spec_nvfp4.evaluate_mapping()
        print("Full NVFP4 mapping evaluation completed.")
        print(nvfp4_full_result)
    except Exception as e:
        print("Full NVFP4 parse succeeded, but evaluate_mapping() failed:")
        print(type(e).__name__, e)
else:
    print("Skipped Spec.from_yaml/evaluate_mapping().")


## Next steps for milestone completion

1. Run all three workloads: **Baseline (FP16)**, **NVFP4 weight-only (W4A16)**, **NVFP4 full (W4A4)**
2. Run `evaluate_mapping()` for each and compare energy/latency
3. Add your quantization compound component and re-evaluate energy/latency
4. Compare the three configurations and report overhead tradeoffs

## Milestone Presentation Visualizations

These plots summarize baseline vs NVFP4-weight-only vs NVFP4-full (W4A4) results for presentation.

- Main chart: normalized energy and latency ($\text{quantized} / \text{baseline}$)
- Optional chart: absolute values (if extractable from mapping objects)
- A compact summary table is also printed

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


def _maybe_get(obj, names):
    for n in names:
        if hasattr(obj, n):
            v = getattr(obj, n)
            if callable(v):
                try:
                    return v()
                except Exception:
                    pass
            else:
                return v
    return None


def extract_metrics(mapping_obj):
    """Best-effort extractor for AccelForge mapping result metrics.
    Returns dict with keys: energy_pj, latency_cycles.
    """
    out = {"energy_pj": None, "latency_cycles": None}
    if mapping_obj is None:
        return out

    # Direct/common fields
    out["energy_pj"] = _maybe_get(mapping_obj, [
        "energy_pj", "energy", "total_energy", "total_energy_pj"
    ])
    out["latency_cycles"] = _maybe_get(mapping_obj, [
        "latency_cycles", "latency", "cycles", "total_latency", "total_cycles"
    ])

    # Some objects expose dataframe-like containers
    for container_name in ["df", "dataframe", "mappings_df", "table"]:
        container = _maybe_get(mapping_obj, [container_name])
        if container is None:
            continue
        try:
            cols = [c.lower() for c in container.columns]
            if out["energy_pj"] is None:
                for c in ["energy_pj", "energy", "total_energy", "total_energy_pj"]:
                    if c in cols:
                        out["energy_pj"] = float(container.loc[:, container.columns[cols.index(c)]].min())
                        break
            if out["latency_cycles"] is None:
                for c in ["latency_cycles", "latency", "cycles", "total_latency", "total_cycles"]:
                    if c in cols:
                        out["latency_cycles"] = float(container.loc[:, container.columns[cols.index(c)]].min())
                        break
        except Exception:
            pass

    # Cast if possible
    for k in ["energy_pj", "latency_cycles"]:
        try:
            if out[k] is not None:
                out[k] = float(out[k])
        except Exception:
            out[k] = None

    return out


# Collect current run results (expects earlier cells to have run)
b = globals().get("baseline_result", None)
q = globals().get("quant_result", None)
nv = globals().get("nvfp4_full_result", None)

baseline_metrics = extract_metrics(b)
quant_metrics = extract_metrics(q)
nvfp4_full_metrics = extract_metrics(nv)

rows = [
    {
        "config": "baseline (FP16)",
        "energy_pj": baseline_metrics["energy_pj"],
        "latency_cycles": baseline_metrics["latency_cycles"],
    },
    {
        "config": "NVFP4 weight-only (W4A16)",
        "energy_pj": quant_metrics["energy_pj"],
        "latency_cycles": quant_metrics["latency_cycles"],
    },
    {
        "config": "NVFP4 full (W4A4)",
        "energy_pj": nvfp4_full_metrics["energy_pj"],
        "latency_cycles": nvfp4_full_metrics["latency_cycles"],
    },
]
summary_df = pd.DataFrame(rows)
display(summary_df)

# Normalized chart if all values are available
all_have_metrics = all(
    m["energy_pj"] is not None and m["latency_cycles"] is not None
    for m in [baseline_metrics, quant_metrics, nvfp4_full_metrics]
)

if all_have_metrics:
    configs = ["W4A16\n(weight-only)", "W4A4\n(NVFP4 full)"]
    norm_energy = [
        quant_metrics["energy_pj"] / baseline_metrics["energy_pj"],
        nvfp4_full_metrics["energy_pj"] / baseline_metrics["energy_pj"],
    ]
    norm_latency = [
        quant_metrics["latency_cycles"] / baseline_metrics["latency_cycles"],
        nvfp4_full_metrics["latency_cycles"] / baseline_metrics["latency_cycles"],
    ]

    import numpy as np
    x = np.arange(len(configs))
    width = 0.35

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Normalized bar chart
    ax = axes[0]
    bars1 = ax.bar(x - width/2, norm_energy, width, label="Energy", color="#4C78A8")
    bars2 = ax.bar(x + width/2, norm_latency, width, label="Latency", color="#F58518")
    ax.axhline(1.0, linestyle="--", color="gray", linewidth=1, label="Baseline")
    ax.set_xticks(x)
    ax.set_xticklabels(configs)
    ax.set_ylabel("Normalized (Quantized / Baseline)")
    ax.set_title("Normalized Energy & Latency vs FP16 Baseline")
    ax.legend()
    for bar_group in [bars1, bars2]:
        for bar in bar_group:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                    f"{bar.get_height():.3f}x", ha="center", va="bottom", fontsize=9)

    # Absolute energy comparison
    ax2 = axes[1]
    abs_labels = ["Baseline\n(FP16)", "W4A16", "W4A4"]
    abs_energy = [baseline_metrics["energy_pj"], quant_metrics["energy_pj"], nvfp4_full_metrics["energy_pj"]]
    colors = ["#72B7B2", "#E45756", "#9D755D"]
    bars = ax2.bar(abs_labels, abs_energy, color=colors)
    ax2.set_title("Absolute Energy (pJ)")
    ax2.set_ylabel("pJ")
    for bar in bars:
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f"{bar.get_height():.0f}", ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    plt.show()
else:
    print("Metrics are not fully extractable from current result objects.")
    print("Presentation fallback: use parse/evaluate success + printed YAMLs + screenshot of mapping success logs.")
    print("If needed, share the exact result object API and I can wire precise metric extraction.")